# Tutorial 9: Train NicheTrans on embryonic mouse brain data

In [ ]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_hd import *
from datasets.data_manager_MISAR_seq import ATAC_RNA_Seq

from utils.utils import *
from utils.utils_training_embryonic_mouse_brain import *
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [7]:
%run ./args/args_MISAR_seq.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.2, dropout_rate=0.1, n_source=3000, workers=4, knn_smooth=True, peak_threshold=0.05, hvg_gene=1500, adata_path='/mnt/datadisk0/Processed_DATA/2023_nm_MISAR_seq', max_epoch=20, stepsize=10, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [8]:
# create the dataloaders
dataset = ATAC_RNA_Seq(peak_threshold=args.peak_threshold, hvg_gene=args.hvg_gene, adata_path=args.adata_path, RNA2ATAC=True, knn_smoothing=args.knn_smooth)
trainloader, testloader = embryonic_mouse_brain(args, dataset)

# create the model
source_dimension, target_dimension = len(dataset.source_panel), len(dataset.target_panel)
model_kwargs = dict(
    source_length=source_dimension,
    target_length=target_dimension,
    noise_rate=args.noise_rate,
    dropout_rate=args.dropout_rate,
    n_spot_types=getattr(dataset, 'n_spot_types', 1),
)
model = NicheTrans(**model_kwargs)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)


------Calculating spatial graph...
The graph contains 17032 edges, 2129 cells.
8.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 14216 edges, 1777 cells.
8.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 15592 edges, 1949 cells.
8.0000 neighbors per cell on average.
source size 1500
target size 34323
=> Spatial atac-rna Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |   3906 spots,
  test     |   1949 spots,
  ------------------------------


KeyboardInterrupt: 

### Initialize loss function (criterion) and optimizer

In [ ]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [ ]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train_binary(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

# test_binary(args, model, testloader)
test_regression(model, testloader, if_sigmoid=True, device=device)
torch.save(model.state_dict(), 'NicheTrans_embryonic_mouse_brain_rna2atac.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 122/122	 Loss 0.624815 (0.669146)
==> Epoch 2/20
Batch 122/122	 Loss 0.578043 (0.601833)
==> Epoch 3/20
Batch 122/122	 Loss 0.547765 (0.563620)
==> Epoch 4/20
Batch 122/122	 Loss 0.521019 (0.536550)
==> Epoch 5/20
Batch 122/122	 Loss 0.504597 (0.521688)
==> Epoch 6/20
Batch 122/122	 Loss 0.502290 (0.507977)
==> Epoch 7/20
Batch 122/122	 Loss 0.504718 (0.499813)
==> Epoch 8/20
Batch 122/122	 Loss 0.482898 (0.491391)
==> Epoch 9/20
Batch 122/122	 Loss 0.491090 (0.482227)
==> Epoch 10/20
Batch 122/122	 Loss 0.457355 (0.475627)
==> Epoch 11/20
Batch 122/122	 Loss 0.470939 (0.472220)
==> Epoch 12/20
Batch 122/122	 Loss 0.473435 (0.472104)
==> Epoch 13/20
Batch 122/122	 Loss 0.458705 (0.470975)
==> Epoch 14/20
Batch 122/122	 Loss 0.463016 (0.469068)
==> Epoch 15/20
Batch 122/122	 Loss 0.477766 (0.468554)
==> Epoch 16/20
Batch 122/122	 Loss 0.464800 (0.467598)
==> Epoch 17/20
Batch 122/122	 Loss 0.473610 (0.467038)
==> Epoch 18/20
Batch 122/122	 Loss 0.452291 (0.466055)
=